In [1]:
from src.helpers import iterate_parallel, normalize_project, get_json_data
from src.metric import TimestampMetric, get_phase

In [2]:
INPUT_FOLDER = "/home/jortvd/thesis-data/rmt/all"
OUTPUT_FOLDER = "../results"

In [3]:
def find_metric(data: dict, metric: str) -> float:
    for item in data.get("measures", []):
        if item.get("metric") == metric:
            return "value" in item and float(item["value"])
    return None

In [4]:
metrics = TimestampMetric("sq_security_issues", True)

def process_item(item):
    project_name, zip_file, timestamp = item
    phase = get_phase(normalize_project(project_name), timestamp)
    data = get_json_data(zip_file)

    rust_value = 0
    other_value = 0

    for component in data.get("components", []):
        if not component.get("qualifier") == "FIL":
            continue
        if component.get("language") == "rust" and phase != "pre":
            rust_value += find_metric(component, "software_quality_security_issues") or 0
        else:
            other_value += find_metric(component, "software_quality_security_issues") or 0

    return [
        (normalize_project(project_name), timestamp, rust_value, True),
        (normalize_project(project_name), timestamp, other_value, False)
    ]

iterate_parallel(process_item, INPUT_FOLDER, "*.json", metrics)
metrics.save(OUTPUT_FOLDER)

100%|██████████| 10225/10225 [00:18<00:00, 559.92it/s]


In [5]:
metrics = TimestampMetric("sq_security_debt_density", True)

def process_item(item):
    project_name, zip_file, timestamp = item
    phase = get_phase(normalize_project(project_name), timestamp)
    data = get_json_data(zip_file)

    rust_debt = 0
    other_debt = 0
    rust_ncloc = 0
    other_ncloc = 0

    for component in data.get("components", []):
        if not component.get("qualifier") == "FIL":
            continue
        if component.get("language") == "rust" and phase != "pre":
            rust_debt += find_metric(component, "software_quality_security_remediation_effort") or 0
            rust_ncloc += find_metric(component, "ncloc") or 0
        else:
            other_debt += find_metric(component, "software_quality_security_remediation_effort") or 0
            other_ncloc += find_metric(component, "ncloc") or 0

    return [
        (normalize_project(project_name), timestamp, rust_debt / (rust_ncloc / 1000) if rust_ncloc > 0 else 0, True),
        (normalize_project(project_name), timestamp, other_debt / (other_ncloc / 1000) if other_ncloc > 0 else 0, False)
    ]

iterate_parallel(process_item, INPUT_FOLDER, "*.json", metrics)
metrics.save(OUTPUT_FOLDER)

100%|██████████| 10225/10225 [00:19<00:00, 523.48it/s]


In [6]:
metrics = TimestampMetric("sq_reliability_debt_density", True)

def process_item(item):
    project_name, zip_file, timestamp = item
    phase = get_phase(normalize_project(project_name), timestamp)
    data = get_json_data(zip_file)

    rust_debt = 0
    other_debt = 0
    rust_ncloc = 0
    other_ncloc = 0

    for component in data.get("components", []):
        if not component.get("qualifier") == "FIL":
            continue
        if component.get("language") == "rust" and phase != "pre":
            rust_debt += find_metric(component, "software_quality_reliability_remediation_effort") or 0
            rust_ncloc += find_metric(component, "ncloc") or 0
        else:
            other_debt += find_metric(component, "software_quality_reliability_remediation_effort") or 0
            other_ncloc += find_metric(component, "ncloc") or 0

    return [
        (normalize_project(project_name), timestamp, rust_debt / (rust_ncloc / 1000) if rust_ncloc > 0 else 0, True),
        (normalize_project(project_name), timestamp, other_debt / (other_ncloc / 1000) if other_ncloc > 0 else 0, False)
    ]

iterate_parallel(process_item, INPUT_FOLDER, "*.json", metrics)
metrics.save(OUTPUT_FOLDER)

100%|██████████| 10225/10225 [00:18<00:00, 553.39it/s]


In [7]:
metrics = TimestampMetric("sq_technical_debt_density", True)

def process_item(item):
    project_name, zip_file, timestamp = item
    phase = get_phase(normalize_project(project_name), timestamp)
    data = get_json_data(zip_file)

    rust_debt = 0
    other_debt = 0
    rust_ncloc = 0
    other_ncloc = 0

    for component in data.get("components", []):
        if not component.get("qualifier") == "FIL":
            continue
        if component.get("language") == "rust" and phase != "pre":
            rust_debt += find_metric(component, "software_quality_maintainability_remediation_effort") or 0
            rust_ncloc += find_metric(component, "ncloc") or 0
        else:
            other_debt += find_metric(component, "software_quality_maintainability_remediation_effort") or 0
            other_ncloc += find_metric(component, "ncloc") or 0

    return [
        (normalize_project(project_name), timestamp, rust_debt / (rust_ncloc / 1000) if rust_ncloc > 0 else 0, True),
        (normalize_project(project_name), timestamp, other_debt / (other_ncloc / 1000) if other_ncloc > 0 else 0, False)
    ]

iterate_parallel(process_item, INPUT_FOLDER, "*.json", metrics)
metrics.save(OUTPUT_FOLDER)

100%|██████████| 10225/10225 [00:17<00:00, 587.64it/s]


In [8]:
metrics = TimestampMetric("sq_duplication", True)

def process_item(item):
    project_name, zip_file, timestamp = item
    phase = get_phase(normalize_project(project_name), timestamp)
    data = get_json_data(zip_file)

    rust_lines = 0
    rust_duplication = 0
    other_lines = 0
    other_duplication = 0

    for component in data.get("components", []):
        if not component.get("qualifier") == "FIL":
            continue
        if component.get("language") == "rust" and phase != "pre":
            rust_duplication += find_metric(component, "duplicated_lines") or 0
            rust_lines += find_metric(component, "ncloc") or 0
        else:
            other_duplication += find_metric(component, "duplicated_lines") or 0
            other_lines += find_metric(component, "ncloc") or 0

    return [
        (normalize_project(project_name), timestamp, (rust_duplication / rust_lines * 100) if rust_lines > 0 else 0, True),
        (normalize_project(project_name), timestamp, (other_duplication / other_lines * 100) if other_lines > 0 else 0, False)
    ]

iterate_parallel(process_item, INPUT_FOLDER, "*.json", metrics)
metrics.save(OUTPUT_FOLDER)

100%|██████████| 10225/10225 [00:17<00:00, 575.15it/s]
